In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, models
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)

])

test_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)

])

In [ ]:
full_train = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
full_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

DOG = 5   # class index for "dog" in CIFAR-10 (cat=3 is our hard negative)

def make_binary(base, seed=42):
    targets = np.array(base.targets)
    dog_idx    = np.where(targets == DOG)[0]
    nondog_idx = np.where(targets != DOG)[0]
    rng = np.random.default_rng(seed)
    nondog_idx = rng.choice(nondog_idx, size=len(dog_idx), replace=False)
    return np.concatenate([dog_idx, nondog_idx])

class BinaryDog(torch.utils.data.Dataset):
    def __init__(self, base, indices):
        self.base, self.indices = base, list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        return img, (1 if label == DOG else 0)   # 1 = dog, 0 = not dog

train_ds = BinaryDog(full_train, make_binary(full_train))
test_ds  = BinaryDog(full_test,  make_binary(full_test))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} images  |  Test: {len(test_ds)} images")

 12%|█▏        | 20.5M/170M [03:01<22:38, 110kB/s]

In [ ]:
model = models.resnet18(weights="IMAGENET1K_V1")

for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable:,} of {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)

    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    print(f"Epoch {epoch+1}: train loss {running/len(train_ds):.3f} | val acc {correct/total:.3f}")




In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
model.eval(); all_preds, all_labels = [], []
with torch.no_grad():
  for imgs, labels in test_loader:
    all_preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
    all_labels.extend(labels.numpy())

    print(confusion_matrix(all_labels, all_preds))
    print(classification_report(all_labels, all_preds, target_names=["not dog", "dog"]))

In [ ]:
from google.colab import files
from PIL import Image
uploaded = files.upload()
for name in uploaded:
  img = Image.open(name).convert("RGB")
  x = test_tf(img).unsqueeze(0).to(device)

  with torch.no_grad():
    probs = torch.softmax(model(x), dim=1) [0]
    print(f"{name}: dog {probs[1]:.1%}  | not dog {probs[0]:.1%}")

In [ ]:
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
uploaded = files.upload()
name = list(uploaded.keys())[0]
img = Image.open(name).convert("RGB")
x = test_tf(img).unsqueeze(0).to(device)
x.requires_grad_(True)
feats, grads = {}, {}
layer = model.layer4[-1]
h1 = layer.register_forward_hook(lambda m, i, o: feats.__setitem__("v", o.detach()))
h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("v", go[0].detach()))
model.eval()
out = model(x)
probs = torch.softmax(out, dim=1)[0]
pred = out.argmax(1)
model.zero_grad()
out[0, pred].backward()
h1.remove(); h2.remove()
w = grads["v"].mean(dim=(2,3), keepdim=True)
cam = F.relu((w * feats["v"]).sum(1)).squeeze()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
cam = F.interpolate(cam[None, None], size=(224, 224), mode="bilinear")[0, 0].cpu().numpy()
base = img.resize((224, 224))
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(base); ax[0].set_title("Your image"); ax[0].axis("off")
ax[1].imshow(base); ax[1].imshow(cam, cmap="jet", alpha=0.5)
ax[1].set_title("Model Analysis"); ax[1].axis("off")
ax[2].barh(["not dog", "dog"], [probs[0].item(), probs[1].item()],
               color=["#888", "#e63946"])
ax[2].set_xlim(0, 1); ax[2].set_title("Confidence")
for i, p in enumerate([probs[0].item(), probs[1].item()]):
  ax[2].text(p + 0.02, i, f"{p:.1%}", va="center")

  plt.tight_layout(); plt.show()
  print(f"{name}: dog {probs[1]:.1%}  |  not dog {probs[0]:.1%}")